In [1]:

    
import numpy as np
nsites = 12
nv = 3
sh = 6
trainfeatures = np.load("trainfeatures.npy")[sh::100, :nv, :].reshape(850, nsites).T
testfeatures = np.load("testfeatures.npy")[sh::100, :nv, :].reshape(120, nsites).T
traintargets = np.load("traintargets.npy")[sh::100, :nv, :].reshape(850, nsites).T
testtargets = np.load("testtargets.npy")[sh::100, :nv, :].reshape(120, nsites).T

#"pauli_prop_values.h5", "w") do file
#    write(file, "qrc_train", expectations)
#    write(file, "qrc_test", expectations_test)
#    write(file, "test_targets", testtargets)
#    write(file, "train_targets", traintargets)
#    write(file, "test_features", testfeatures)
#    write(file, "train_features", trainfeatures)

In [2]:
import numpy as np
statsfeatures = np.zeros((970, nsites))
allpredict = np.zeros((970, nsites))
statsfeatures[:850,:] = trainfeatures.T
statsfeatures[850:,:] = testfeatures.T
allpredict[:850,:] = traintargets.T
allpredict[850:,:] = testtargets.T

In [4]:
train_embeddings = np.load("train_embeddings_noiseless.npy")
test_embeddings = np.load("test_embeddings_noiseless.npy")
#train_embeddings = np.load("train_embeddings_noisy.npy")
#test_embeddings = np.load("test_embeddings_noisy.npy")
#train_embeddings = np.load("train_embeddings_noisy2.npy")
#test_embeddings = np.load("test_embeddings_noisy2.npy")
train_embeddings = np.load("train_embeddings_noisy4.npy")
test_embeddings = np.load("test_embeddings_noisy4.npy")
#train_embeddings = np.load("train_embeddings_noiselessl.npy")
#test_embeddings = np.load("test_embeddings_noiselessl.npy")
train_embeddings.shape

(850, 780)

In [5]:
#train_embeddings = np.load("train_embeddings_noiseless.npy")
#test_embeddings = np.load("test_embeddings_noiseless.npy")
#test_embeddingsn = np.load("test_embeddings_noiselessl.npy")
#print(test_embeddingsn[:,2])
#print(test_embeddings[:,2])
quantum_train = np.zeros((850,360))
quantum_test = np.zeros((120,360))

quantum_train = train_embeddings
quantum_test = test_embeddings[:,:]

In [6]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

all_r2 = np.zeros(12)
all_mse = np.zeros(12)
all_r2_qrc = np.zeros(12)
all_mse_qrc = np.zeros(12)
imps = []
ma = []
mb = []
n_windows = 1
bs = 3
ws = test_embeddings.shape[1]//10
for l in range(nsites):
    # Train simle linear
    X_Train = statsfeatures[:850]
    X_Test = statsfeatures[850:]
    Y_Train = allpredict[:850,l] #np.concatenate((traintargets[:,i,:], traintargets[:,j,:], traintargets[:,k,:]), axis=1)[:,:]
    Y_Test = allpredict[850:,l] #np.concatenate((testtargets[:,i,:], testtargets[:,j,:], testtargets[:,k,:]), axis=1)[:,:]
    alphas = np.logspace(-3, 4, 40)   # 100 candidates from 0.0001 to 10000
    from sklearn.linear_model import Ridge, RidgeCV
    ridge_cv = RidgeCV(
            alphas=alphas,
            cv=10,                          # 10-fold cross-validation
            scoring="neg_mean_squared_error"
)
    ridge_cv.fit(X_Train, Y_Train)
    alpha_best = ridge_cv.alpha_
    ridge_model = Ridge(alpha=alpha_best, tol=1.e-4, max_iter=8000) # You can adjust alpha
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_Train)
    X_test_scaled = scaler.transform(X_Test)
    ridge_model.fit(X_train_scaled, Y_Train)
    y_pred = ridge_model.predict(X_test_scaled)
    mse = mean_squared_error(Y_Test, y_pred)
    r2 = r2_score(Y_Test, y_pred)
    all_r2[l] = r2
    all_mse[l] = mse 
    alpha_best1 = alpha_best
    #print(X_Train.shape)
    
    # Train QRC
    X_Train = np.concatenate((X_Train[:], quantum_train[:850,ws*bs+12:ws*bs+ws*n_windows-0]), axis=1)#, quantum_train[:,-ws*n_windows:]), axis=1)
    #print(X_Train.shape)
    X_Test = np.concatenate((X_Test, quantum_test[:,ws*bs+12:ws*bs+ws*n_windows-0]), axis=1)#, quantum_test[:,-ws*n_windows:]), axis=1)
    ridge_cv = RidgeCV(
            alphas=alphas,
            cv=10,                          # 10-fold cross-validation
            scoring="neg_mean_squared_error"
    )
    
    ridge_cv.fit(X_Train, Y_Train)
    alpha_best = ridge_cv.alpha_
    #print(alpha_best1, alpha_best)
    ridge_model = Ridge(alpha=alpha_best, tol=1.e-4, max_iter=8000) # You can adjust alpha
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_Train)
    X_test_scaled = scaler.transform(X_Test)
    ridge_model.fit(X_train_scaled, Y_Train[:])
    y_pred = ridge_model.predict(X_test_scaled)
    mse = mean_squared_error(Y_Test, y_pred)
    r2 = r2_score(Y_Test, y_pred)
    all_r2_qrc[l] = r2
    all_mse_qrc[l] = mse
    if all_r2[l] > -0.5 and ((l < 12) or (l > 35 and l < (36+12))):        
        print(f"{l} {all_r2[l]}          {all_r2_qrc[l]}       {all_r2[l]-all_r2_qrc[l]}")
        imps += [all_r2[l]-all_r2_qrc[l]]
        ma += [all_r2[l]]
        mb += [all_r2_qrc[l]]
    
# print(np.mean(imps), np.std(imps), np.mean(ma), np.mean(mb))

0 0.9182248669986668          0.9131071122990292       0.0051177546996376
1 0.4242641446714702          0.3918917100136132       0.032372434657857
2 0.34199778373795886          0.1892518704954267       0.15274591324253217
3 -0.1545078028330853          -0.7279092335148558       0.5734014306817705
4 -0.13388481372077465          0.02951313751455209       -0.16339795123532674
5 0.8063301402817006          0.7797558792584229       0.026574261023277645
6 0.014962387563615676          0.00993288495357092       0.005029502610044756
7 0.12272305772040926          0.26786938793366943       -0.14514633021326018
8 0.8461763802624647          0.8126271128297926       0.03354926743267206
9 0.7277098013735392          0.6933117666914919       0.034398034682047385
10 0.4538132955182309          0.4641848372007389       -0.010371541682507979
11 -0.20142216777319955          -0.5610292380829032       0.3596070703097036
